In [1]:
from datetime import date
from pathlib import Path

from conf.behavior_cloning.diffusion.five_demos.default import config
from tapas_gmm.dataset.scene import SceneDataset
from tapas_gmm.dataset.bc import BCDataset
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy
from tapas_gmm.behavior_cloning import run_training
from tapas_gmm.utils.select_gpu import device

2026-07-20 23:09:35.633 | INFO     |  Running on cpu


In [2]:
artifact_root = Path("../artifacts")
data_root = artifact_root / "datasets/tapas_diffusion/bimanual_dual_push_buttons"
checkpoint_dir = artifact_root / "checkpoints/diffusion/joint" / date.today().isoformat()
checkpoint_dir.mkdir(parents=True, exist_ok=True)
horizon = 16
n_obs_steps = 2
n_action_steps = 8
epochs = 1500

In [3]:
config.policy.action_dim = 16
config.policy.obs_dim = 35
config.policy.horizon = horizon
config.policy.n_obs_steps = n_obs_steps
config.policy.n_action_steps = n_action_steps
config.policy.training.lr_num_epochs = epochs
config.policy.obs_encoder = ObservationEncoderConfig(
    ee_pose=True,
    object_poses=True,
)

config.policy.unet.input_dim = 16
config.policy.unet.global_cond_dim = 35 * n_obs_steps

In [4]:
config.bc_data.fragment_length = horizon + 1
config.bc_data.pre_padding = n_obs_steps - 1
config.bc_data.post_padding = n_action_steps - 1
config.bc_data.cameras = tuple()

In [5]:
config.training.epochs = epochs
config.training.save_freq = 100

In [7]:
loaded_dataset = SceneDataset(data_root=data_root)

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=config.bc_data,
)

policy = DiffusionPolicy(config.policy).to(device)

2026-07-20 23:09:42.484 | INFO     |  Initializing datasete using ../outputs/bimanual_dataset/metadata.json
2026-07-20 23:09:42.570 | INFO     |  Extracted gt object labels []
2026-07-20 23:09:42.571 | INFO     |  Extracted tsdf object labels []
2026-07-20 23:09:42.571 | INFO     |  Initializing BCDataset:
2026-07-20 23:09:42.571 | INFO     |    Training on fragments of length 17.
2026-07-20 23:09:42.571 | INFO     |    Loading raw data for encoder.
2026-07-20 23:09:42.571 | INFO     |  Initializing DiffusionPolicy:
2026-07-20 23:09:42.571 | INFO     |    Initializing Policy:
2026-07-20 23:09:42.814 | INFO     |    number of parameters: 5525968
2026-07-20 23:09:42.882 | INFO     |    No encoder config provided. Using None.
None


In [8]:
import wandb
wandb.init(mode="disabled")

run_training(
    policy,
    bc_dataset,
    config,
    str(checkpoint_dir / "joint"),
)

2026-07-20 23:09:43.190 | INFO     |  No datasplit specified.
2026-07-20 23:09:43.192 | INFO     |    Setting action scaling for optimal policy performance. Using DP normalizer implementation.
2026-07-20 23:09:45.392 | INFO     |  Beginning training.


  0%|          | 0/500 [00:00<?, ?it/s]

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


2026-07-20 23:44:55.106 | INFO     |  Saving intermediate policy:
2026-07-20 23:44:55.108 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_100.pt
2026-07-20 23:44:55.136 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_100_ema.pt
2026-07-21 00:27:30.666 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_200.pt
2026-07-21 00:27:30.697 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_200_ema.pt
2026-07-21 00:58:35.010 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_300.pt
2026-07-21 00:58:35.086 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_300_ema.pt
2026-07-21 01:22:07.776 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_400.pt
2026-07-21 01:22:07.801 | INFO     |    Saving policy at ../outputs/bimanual_diffusion_policy_step_400_ema.pt
2026-07-21 01:45:43.608 | INFO     |    Saving policy at ../outputs/bi

In [9]:
policy.to_disk(str(checkpoint_dir / "final.pt"))
(checkpoint_dir / f"joint_step_{epochs}_ema.pt").replace(
    checkpoint_dir / "latest.pt"
)

2026-07-21 01:45:43.783 | INFO     |  Saving policy at ../outputs/bimanual_diffusion_policy.pt
